# Exact verification of a proposed counterexample to the Jacobian conjecture

Consider the polynomial map $F=(F_1,F_2,F_3):\mathbb{C}^3\to\mathbb{C}^3$ defined by

$$\begin{aligned}
F_1(x,y,z)&=(1+xy)^3z+y^2(1+xy)(4+3xy),\\
F_2(x,y,z)&=y+3x(1+xy)^2z+3xy^2(4+3xy),\\
F_3(x,y,z)&=2x-3x^2y-x^3z.
\end{aligned}$$

This notebook uses exact symbolic arithmetic to check that $\det J_F=-2$ and that three distinct points have the same image $(-1/4,0,0)$. If both claims hold, then $F$ is a noninjective polynomial map with nonzero constant Jacobian determinant, which is precisely a counterexample to the complex Jacobian conjecture in dimension 3.

In [ ]:
import sympy as sp
from IPython.display import display

sp.init_printing()
x, y, z = sp.symbols('x y z')

F1 = (1 + x*y)**3*z + y**2*(1 + x*y)*(4 + 3*x*y)
F2 = y + 3*x*(1 + x*y)**2*z + 3*x*y**2*(4 + 3*x*y)
F3 = 2*x - 3*x**2*y - x**3*z
F = sp.Matrix([F1, F2, F3])
F

## 1. Compute the Jacobian determinant

In [ ]:
variables = (x, y, z)
J = F.jacobian(variables)
display(J)

det_J_unexpanded = J.det(method='berkowitz')
det_J = sp.factor(det_J_unexpanded)
print('Jacobian determinant:', det_J)
assert sp.expand(det_J + 2) == 0
det_J

The assertion verifies the polynomial identity $\det J_F+2=0$ exactly. Thus the Jacobian determinant is the nonzero constant $-2$ everywhere on $\mathbb{C}^3$.

## 2. Evaluate the three proposed preimages

In [ ]:
points = [
    (sp.Integer(0), sp.Integer(0), -sp.Rational(1, 4)),
    (sp.Integer(1), -sp.Rational(3, 2), sp.Rational(13, 2)),
    (-sp.Integer(1), sp.Rational(3, 2), sp.Rational(13, 2)),
]
target = sp.Matrix([-sp.Rational(1, 4), 0, 0])

def evaluate_map(point):
    substitution = dict(zip(variables, point))
    return F.applyfunc(lambda expression: sp.factor(expression.subs(substitution)))

images = [evaluate_map(point) for point in points]
for index, (point, image) in enumerate(zip(points, images), start=1):
    print(f'P{index} = {point}  -->  {tuple(image)}')
    assert image == target

assert len(set(points)) == 3
print('All three distinct points map to', tuple(target))

## 3. Conclusion

The exact computations establish both required facts:

1. $\det J_F=-2\ne0$ identically.
2. The three distinct points
   $$\left(0,0,-\frac14\right),\quad\left(1,-\frac32,\frac{13}{2}\right),\quad\left(-1,\frac32,\frac{13}{2}\right)$$
   all map to $(-1/4,0,0)$.

Consequently $F$ is not injective. A polynomial automorphism must be injective, so $F$ has no polynomial inverse. Therefore, assuming the displayed formula is the claimed map, this computation verifies a counterexample to the Jacobian conjecture over $\mathbb{C}$ in dimension 3.

## Optional compact verification

The following cell repeats the essential checks in a few lines and returns `True`.

In [ ]:
verified = (
    sp.expand(F.jacobian((x, y, z)).det() + 2) == 0
    and len(set(points)) == 3
    and all(evaluate_map(point) == target for point in points)
)
verified